# 1. Import Libraries
We import pandas for data manipulation and scikit-learn for building and evaluating our machine learning model.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

import joblib

# 2. Load Data
We load the datasets required for building the predictive model.

In [2]:
attendance = pd.read_csv("../data/attendance.csv")

quiz = pd.read_csv("../data/quiz_attempts.csv")

mock = pd.read_csv("../data/mock_tests.csv")

assignment = pd.read_csv("../data/assignments.csv")

engagement = pd.read_csv("../data/engagement.csv")

In [3]:
attendance = pd.read_csv("../data/attendance.csv")

quiz = pd.read_csv("../data/quiz_attempts.csv")

mock = pd.read_csv("../data/mock_tests.csv")

assignment = pd.read_csv("../data/assignments.csv")

engagement = pd.read_csv("../data/engagement.csv")

# 3. Data Aggregation and Merging
We calculate the average metrics (attendance, quiz accuracy, mock test marks, assignment scores) per student and merge them into a single consolidated DataFrame along with engagement metrics.

In [4]:
attendance_avg = attendance.groupby("Student_ID")["Attendance_Percentage"].mean().reset_index()

In [5]:
quiz_avg = quiz.groupby("Student_ID")["Accuracy"].mean().reset_index()

In [6]:
mock_avg = mock.groupby("Student_ID")["Marks"].mean().reset_index()

In [7]:
assignment_avg = assignment.groupby("Student_ID")["Score"].mean().reset_index()

In [8]:
data = attendance_avg.merge(
    quiz_avg,
    on="Student_ID"
)

data = data.merge(
    mock_avg,
    on="Student_ID"
)

data = data.merge(
    assignment_avg,
    on="Student_ID"
)

data = data.merge(
    engagement,
    on="Student_ID"
)

data.head()

,Student_ID,Attendance_Percentage,Accuracy,Marks,Score,Login_Frequency,Weekly_Study_Hours,Average_Session_Duration,Consecutive_Study_Days,Sessions_Per_Week
0,STU0001,80.8,37.736667,69.333333,65.0,18,16.3,35,6,4
1,STU0002,67.4,61.533333,83.000000,40.8,14,23.5,52,7,9
2,STU0003,71.1,67.313333,71.000000,48.8,16,9.7,69,25,4
3,STU0004,66.7,82.231667,60.666667,33.4,13,19.8,169,17,2
4,STU0005,81.7,64.335000,38.666667,41.8,20,14.3,144,7,4


# 4. Define Target Variable (Performance)
We define the target variable 'Performance' based on mock test marks to classify students into 'High Performer', 'Average', or 'At Risk'.

In [9]:
def performance(row):

    if row["Marks"] >= 80:
        return "High Performer"

    elif row["Marks"] >= 50:
        return "Average"

    else:
        return "At Risk"

In [10]:
data["Performance"] = data.apply(
    performance,
    axis=1
)

In [11]:
data.head()

,Student_ID,Attendance_Percentage,Accuracy,Marks,Score,Login_Frequency,Weekly_Study_Hours,Average_Session_Duration,Consecutive_Study_Days,Sessions_Per_Week,Performance
0,STU0001,80.8,37.736667,69.333333,65.0,18,16.3,35,6,4,Average
1,STU0002,67.4,61.533333,83.000000,40.8,14,23.5,52,7,9,High Performer
2,STU0003,71.1,67.313333,71.000000,48.8,16,9.7,69,25,4,Average
3,STU0004,66.7,82.231667,60.666667,33.4,13,19.8,169,17,2,Average
4,STU0005,81.7,64.335000,38.666667,41.8,20,14.3,144,7,4,At Risk


# 5. Feature Selection
We separate the features (X) which include various academic and engagement metrics, and the target label (y).

In [12]:
# Rename columns to match final insights
data.rename(columns={
    "Accuracy": "Quiz_Average",
    "Score": "Assignment_Average"
}, inplace=True)

# Create an Engagement_Score
data["Engagement_Score"] = (data["Login_Frequency"] * 2 + data["Weekly_Study_Hours"] * 3).clip(0, 100)

data.head()


,Student_ID,Attendance_Percentage,Quiz_Average,Marks,Assignment_Average,Login_Frequency,Weekly_Study_Hours,Average_Session_Duration,Consecutive_Study_Days,Sessions_Per_Week,Performance,Engagement_Score
0,STU0001,80.8,37.736667,69.333333,65.0,18,16.3,35,6,4,Average,84.9
1,STU0002,67.4,61.533333,83.000000,40.8,14,23.5,52,7,9,High Performer,98.5
2,STU0003,71.1,67.313333,71.000000,48.8,16,9.7,69,25,4,Average,61.1
3,STU0004,66.7,82.231667,60.666667,33.4,13,19.8,169,17,2,Average,85.4
4,STU0005,81.7,64.335000,38.666667,41.8,20,14.3,144,7,4,At Risk,82.9


In [13]:
X = data[
    [
        "Attendance_Percentage",
        "Quiz_Average",
        "Marks",
        "Assignment_Average",
        "Weekly_Study_Hours",
        "Average_Session_Duration",
        "Engagement_Score",
        "Sessions_Per_Week"
    ]
]

y = data["Performance"]

# 6. Train-Test Split
We split the dataset into training and testing sets, allocating 20% of the data for testing the model's performance.

In [14]:
X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.2,

    random_state=42
)

# 7. Model Training
We initialize and train a Random Forest Classifier using our training data.

In [15]:
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


# 8. Prediction
We use the trained model to predict the performance categories on the test set.

In [16]:
predictions = model.predict(
    X_test
)

# 9. Model Evaluation
We evaluate the model using Accuracy Score, Classification Report, and Confusion Matrix to understand its precision, recall, and overall performance.

In [17]:
accuracy = accuracy_score(
    y_test,
    predictions
)

print("Accuracy :", accuracy)

Accuracy : 1.0


In [18]:
print(

classification_report(

y_test,

predictions

)

)

                precision    recall  f1-score   support

       At Risk       1.00      1.00      1.00        97
       Average       1.00      1.00      1.00       275
High Performer       1.00      1.00      1.00        28

      accuracy                           1.00       400
     macro avg       1.00      1.00      1.00       400
  weighted avg       1.00      1.00      1.00       400



In [19]:
print(

confusion_matrix(

y_test,

predictions

)

)

[[ 97   0   0]
 [  0 275   0]
 [  0   0  28]]


In [20]:
import os
import joblib

os.makedirs('../models', exist_ok=True)
joblib.dump(model, '../models/student_model.pkl')
features_to_save = [
    "Attendance_Percentage",
    "Quiz_Average",
    "Marks",
    "Assignment_Average",
    "Weekly_Study_Hours",
    "Average_Session_Duration",
    "Engagement_Score",
    "Sessions_Per_Week",
    "Performance"
]
data[features_to_save].to_csv('../data/student_performance.csv', index=False)
